In [14]:
import numpy as np
import pandas as pd

states = ['D', 'M', 'U']
observations_vocab = ['FB', 'B', 'S', 'NS']
obs_map = {obs: idx for idx, obs in enumerate(observations_vocab)}

# a)
pi = np.array([1/3, 1/3, 1/3])

A = np.array([
    [0.00, 0.50, 0.50],
    [0.50, 0.25, 0.25],
    [0.5, 0.25, 0.25]
])

B = np.array([
    [0.10, 0.20, 0.40, 0.30],
    [0.15, 0.25, 0.50, 0.10],
    [0.20, 0.30, 0.40, 0.10]
])

obs_seq = ['FB', 'FB', 'S', 'B', 'B', 'S', 'B', 'B', 'NS', 'B', 'B', 'S']
obs_indices = [obs_map[o] for o in obs_seq]

print("Distributia initială pi:", np.round(pi, 4))
print("\nMatricea de tranzitie A:\n", pd.DataFrame(A, index=states, columns=states))
print("\nMatricea de emisie B:\n", pd.DataFrame(B, index=states, columns=observations_vocab))

print("\nVerificare sume")
print("Suma pi:", np.sum(pi))
print("Sumele pe randuri in A:", np.sum(A, axis=1))
print("Sumele pe randuri in B:", np.sum(B, axis=1))

Distributia initială pi: [0.3333 0.3333 0.3333]

Matricea de tranzitie A:
      D     M     U
D  0.0  0.50  0.50
M  0.5  0.25  0.25
U  0.5  0.25  0.25

Matricea de emisie B:
      FB     B    S   NS
D  0.10  0.20  0.4  0.3
M  0.15  0.25  0.5  0.1
U  0.20  0.30  0.4  0.1

Verificare sume
Suma pi: 1.0
Sumele pe randuri in A: [1. 1. 1.]
Sumele pe randuri in B: [1. 1. 1.]


In [15]:
def viterbi_custom(pi, A, B, obs_indices):
    T = len(obs_indices)
    N = len(pi)
    
    delta = np.zeros((T, N), dtype=float)
    psi = np.zeros((T, N), dtype=int)
    
    # b)
    o_1 = obs_indices[0]
    for j in range(N):
        delta[0, j] = pi[j] * B[j, o_1]
        psi[0, j] = 0
        
    # c)
    for t in range(1, T):
        o_t = obs_indices[t]
        for j in range(N):
            transition_scores = [delta[t - 1, i] * A[i, j] for i in range(N)]

            max_score = np.max(transition_scores)
            best_prev_state = int(np.argmax(transition_scores))
            
            delta[t, j] = B[j, o_t] * max_score
            psi[t, j] = best_prev_state
            
    return delta, psi

delta, psi = viterbi_custom(pi, A, B, obs_indices)

In [16]:
#d
print("Valorile delta_1 (dupa primul FB):")
for j, s in enumerate(states):
    print(f"  delta_1({s}) = {delta[0, j]:.6f}")

print("\nValorile delta_2 (dupa al doilea FB):")
for j, s in enumerate(states):
    prev_state = states[psi[1, j]]
    print(f"  delta_2({s}) = {delta[1, j]:.6f}  (stare precedenta optima psi_2({s}) = {prev_state})")

Valorile delta_1 (dupa primul FB):
  delta_1(D) = 0.033333
  delta_1(M) = 0.050000
  delta_1(U) = 0.066667

Valorile delta_2 (dupa al doilea FB):
  delta_2(D) = 0.003333  (stare precedenta optima psi_2(D) = U)
  delta_2(M) = 0.002500  (stare precedenta optima psi_2(M) = D)
  delta_2(U) = 0.003333  (stare precedenta optima psi_2(U) = D)


In [17]:
# e)
T = len(obs_indices)
best_path_indices = [0] * T

best_path_indices[T - 1] = int(np.argmax(delta[T - 1, :]))
max_joint_prob = float(np.max(delta[T - 1, :]))

for t in range(T - 2, -1, -1):
    best_path_indices[t] = psi[t + 1, best_path_indices[t + 1]]

best_path_states = [states[idx] for idx in best_path_indices]

df_results = pd.DataFrame(
    delta, 
    columns=[f"delta({s})" for s in states],
    index=[f"t={t+1} ({obs_seq[t]})" for t in range(T)]
)
df_results["Stare Viterbi"] = best_path_states
print("Evolutia scorurilor Viterbi si traseul reconstituit:\n")
print(df_results.to_string())

print("\nObservatii (Note):         ", " -> ".join(obs_seq))
print("Secventa Viterbi (Stari):  ", " -> ".join(best_path_states))
print(f"Probabilitatea comuna max P(x*, o): {max_joint_prob:.12e}")


Evolutia scorurilor Viterbi si traseul reconstituit:

              delta(D)      delta(M)      delta(U) Stare Viterbi
t=1 (FB)  3.333333e-02  5.000000e-02  6.666667e-02             D
t=2 (FB)  3.333333e-03  2.500000e-03  3.333333e-03             U
t=3 (S)   6.666667e-04  8.333333e-04  6.666667e-04             D
t=4 (B)   8.333333e-05  8.333333e-05  1.000000e-04             U
t=5 (B)   1.000000e-05  1.041667e-05  1.250000e-05             D
t=6 (S)   2.500000e-06  2.500000e-06  2.000000e-06             M
t=7 (B)   2.500000e-07  3.125000e-07  3.750000e-07             D
t=8 (B)   3.750000e-08  3.125000e-08  3.750000e-08             U
t=9 (NS)  5.625000e-09  1.875000e-09  1.875000e-09             D
t=10 (B)  1.875000e-10  7.031250e-10  8.437500e-10             U
t=11 (B)  8.437500e-11  5.273438e-11  6.328125e-11             D
t=12 (S)  1.265625e-11  2.109375e-11  1.687500e-11             M

Observatii (Note):          FB -> FB -> S -> B -> B -> S -> B -> B -> NS -> B -> B -> S
Secventa Vit